# Réentraînement EfficientNet B0 - 30 Epochs
Pour améliorer la confiance du modèle

In [ ]:
# 1. Installation
!pip install torch torchvision pillow -q
print("✅ Installation terminée")

In [ ]:
# 2. Upload le dataset FracAtlas
from google.colab import files
import zipfile
import os

print("📦 Compresse ton dossier FracAtlas en ZIP")
print("   C:\\Users\\Hanae\\Downloads\\FracAtlas")
print("\n⬆️ Upload le ZIP:")
uploaded = files.upload()
print("\n✅ Upload terminé")

In [ ]:
# 3. Extraction
zip_filename = list(uploaded.keys())[0]
print(f"📂 Extraction de {zip_filename}...")

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('/content/')

print("✅ Extraction terminée")

In [ ]:
# 4. Organiser les données (80% train, 10% val, 10% test)
import shutil
from pathlib import Path
import random

# Trouver le dossier FracAtlas
base = '/content/FracAtlas' if os.path.exists('/content/FracAtlas') else '/content'

# Créer structure
for split in ['train', 'val', 'test']:
    for cat in ['Fractured', 'Not_Fractured']:
        os.makedirs(f'/content/dataset/{split}/{cat}', exist_ok=True)

# Organiser les images
random.seed(42)

for category in ['Fractured', 'Non_fractured']:
    src_dir = f'{base}/{category}'
    if not os.path.exists(src_dir):
        continue
    
    images = list(Path(src_dir).glob('*.jpg')) + list(Path(src_dir).glob('*.png'))
    random.shuffle(images)
    
    total = len(images)
    train_end = int(0.8 * total)
    val_end = int(0.9 * total)
    
    dest_cat = 'Fractured' if category == 'Fractured' else 'Not_Fractured'
    
    # Train
    for img in images[:train_end]:
        shutil.copy(img, f'/content/dataset/train/{dest_cat}/{img.name}')
    
    # Val
    for img in images[train_end:val_end]:
        shutil.copy(img, f'/content/dataset/val/{dest_cat}/{img.name}')
    
    # Test
    for img in images[val_end:]:
        shutil.copy(img, f'/content/dataset/test/{dest_cat}/{img.name}')

# Vérifier
for split in ['train', 'val', 'test']:
    f = len(os.listdir(f'/content/dataset/{split}/Fractured'))
    nf = len(os.listdir(f'/content/dataset/{split}/Not_Fractured'))
    print(f"✅ {split}: {f} Fractured, {nf} Not_Fractured")

In [ ]:
# 5. Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

In [ ]:
# 6. Data Augmentation & Loaders
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder('/content/dataset/train', transform=train_transform)
val_dataset = datasets.ImageFolder('/content/dataset/val', transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"✅ Train: {len(train_dataset)} images")
print(f"✅ Val: {len(val_dataset)} images")

In [ ]:
# 7. Modèle
class BoneFractureClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.efficientnet_b0(weights='IMAGENET1K_V1')
        self.features = backbone.features
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

model = BoneFractureClassifier().to(device)
print("✅ Modèle créé")

In [ ]:
# 8. Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3)

print("✅ Setup terminé")

In [ ]:
# 9. Entraînement 30 epochs
epochs = 30
best_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("🚀 Début entraînement - 30 epochs\n")

for epoch in range(epochs):
    # TRAIN
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
    
    train_acc = 100. * correct / total
    
    # VALIDATION
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    val_acc = 100. * correct / total
    
    # Sauvegarder le meilleur
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
        }, '/content/best_model_extended.pth')
        print(f"\n💾 Nouveau meilleur modèle sauvegardé! Acc: {val_acc:.2f}%\n")
    
    scheduler.step(val_acc)
    
    print(f"Epoch {epoch+1}: Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
    print("-" * 60)

print(f"\n✅ Entraînement terminé!")
print(f"🏆 Meilleure précision: {best_acc:.2f}%")

In [ ]:
# 10. Télécharger le modèle
print("📥 Téléchargement du modèle...")
files.download('/content/best_model_extended.pth')
print("\n✅ Téléchargement terminé!")
print("\n📌 IMPORTANT:")
print("   1. Renomme le fichier en: best_model.pth")
print("   2. Remplace l'ancien dans: models/best_model.pth")